In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv('../data/bitcoin_historical_data.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
df = df.sort_values('Timestamp').reset_index(drop=True)

# Calculate indicators (from Week 2)
df['MA_20'] = df['Close'].rolling(window=20).mean()
df['MA_50'] = df['Close'].rolling(window=50).mean()
df['Daily_Return'] = df['Close'].pct_change() * 100

def calculate_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI'] = calculate_rsi(df['Close'])

# Create target variable: Will price go UP or DOWN tomorrow?
df['Price_Tomorrow'] = df['Close'].shift(-1)
df['Target'] = (df['Price_Tomorrow'] > df['Close']).astype(int)  # 1 = up, 0 = down

# Drop rows with NaN values (from calculations)
df = df.dropna()

print(f"✅ Data shape after cleaning: {df.shape}")
print(f"✅ Rows: {len(df)}, Columns: {len(df.columns)}")

# Select features for the model
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'MA_20', 'MA_50', 'Daily_Return', 'RSI']
X = df[features]
y = df['Target']

print(f"\n✅ Features selected: {features}")
print(f"✅ Target variable: Predict if price goes UP (1) or DOWN (0)")
print(f"\nTarget distribution:")
print(y.value_counts())

# Normalize features (scale them to 0-1 range)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=features)

# Split data: 80% train, 20% test
# IMPORTANT: Don't shuffle! Time series data must stay in order
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, shuffle=False
)

print(f"\n✅ Data split:")
print(f"   Training set: {len(X_train)} samples")
print(f"   Test set: {len(X_test)} samples")

print(f"\n✅ Training period: {df['Timestamp'].iloc[0]} to {df['Timestamp'].iloc[len(X_train)-1]}")
print(f"✅ Test period: {df['Timestamp'].iloc[len(X_train)]} to {df['Timestamp'].iloc[-1]}")

# Save for next notebook
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("\n✅ Data saved! Ready for model training.")

✅ Data shape after cleaning: (6969428, 12)
✅ Rows: 6969428, Columns: 12

✅ Features selected: ['Open', 'High', 'Low', 'Close', 'Volume', 'MA_20', 'MA_50', 'Daily_Return', 'RSI']
✅ Target variable: Predict if price goes UP (1) or DOWN (0)

Target distribution:
Target
0    4189287
1    2780141
Name: count, dtype: int64

✅ Data split:
   Training set: 5575542 samples
   Test set: 1393886 samples

✅ Training period: 2012-01-01 20:28:00 to 2023-06-30 18:27:00
✅ Test period: 2023-06-30 18:28:00 to 2026-02-24 00:00:00

✅ Data saved! Ready for model training.
